# Day 02 — Visual Diagrams
**Matrix Multiplication · Linear Transformations · Determinants · Change of Basis**

Run each cell to see the diagram. All diagrams use matplotlib only.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

## Diagram 1 — Linear Transformation: Unit Square Under a Matrix
Shows how a 2×2 matrix transforms the unit square. The columns of A are where the basis vectors land.

In [ ]:
def draw_grid(ax, M, color, label, alpha=0.3):
    """Draw a transformed grid."""
    lines = np.linspace(-2, 2, 9)
    for x in lines:
        pts = np.array([[x, y] for y in np.linspace(-2, 2, 50)]).T
        transformed = M @ pts
        ax.plot(transformed[0], transformed[1], color=color, alpha=alpha, lw=0.6)
    for y in lines:
        pts = np.array([[x, y] for x in np.linspace(-2, 2, 50)]).T
        transformed = M @ pts
        ax.plot(transformed[0], transformed[1], color=color, alpha=alpha, lw=0.6)

def draw_unit_square(ax, M, color, label):
    corners = np.array([[0,0],[1,0],[1,1],[0,1],[0,0]]).T
    transformed = M @ corners
    ax.fill(transformed[0], transformed[1], alpha=0.25, color=color)
    ax.plot(transformed[0], transformed[1], color=color, lw=2, label=label)

def draw_basis_vectors(ax, M, colors=('tab:red','tab:blue')):
    for i, c in enumerate(colors):
        v = M[:, i]
        ax.annotate('', xy=v, xytext=(0,0),
                    arrowprops=dict(arrowstyle='->', color=c, lw=2.5))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Linear Transformations: Unit Square & Basis Vectors', fontsize=13, fontweight='bold')

transforms = [
    (np.eye(2),                        'Identity',           'gray'),
    (np.array([[1, 0.8],[0, 1]]),       'Shear (k=0.8)',      'tab:green'),
    (np.array([[np.cos(np.pi/4), -np.sin(np.pi/4)],
               [np.sin(np.pi/4),  np.cos(np.pi/4)]]), 'Rotation 45°', 'tab:purple'),
]

for ax, (M, title, col) in zip(axes, transforms):
    draw_grid(ax, M, col, title)
    draw_unit_square(ax, M, col, 'transformed square')
    draw_basis_vectors(ax, M)
    ax.set_xlim(-1.5, 2.5); ax.set_ylim(-1.5, 2.5)
    ax.axhline(0, color='black', lw=0.8); ax.axvline(0, color='black', lw=0.8)
    ax.set_aspect('equal'); ax.grid(False)
    ax.set_title(f'{title}\nMatrix = {M.tolist()}', fontsize=9)
    col1 = M[:,0]; col2 = M[:,1]
    ax.annotate(f'col1={col1}', xy=col1, fontsize=8, color='tab:red',
                xytext=(col1[0]+0.1, col1[1]+0.1))
    ax.annotate(f'col2={col2}', xy=col2, fontsize=8, color='tab:blue',
                xytext=(col2[0]+0.1, col2[1]+0.1))

plt.tight_layout()
plt.savefig('diagram1_transformations.png', bbox_inches='tight')
plt.show()
print('KEY: Red arrow = where e₁=[1,0] lands | Blue arrow = where e₂=[0,1] lands')

## Diagram 2 — Composition of Transformations: AB ≠ BA
Shows that rotation then shear ≠ shear then rotation — the order of matrix multiplication matters.

In [ ]:
theta = np.pi / 4
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
S = np.array([[1, 1.0], [0, 1]])

AB = R @ S   # shear first, then rotate
BA = S @ R   # rotate first, then shear

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
fig.suptitle('Composition: AB ≠ BA  (Order Matters!)', fontsize=13, fontweight='bold')

configs = [
    (np.eye(2), 'Original',           'gray'),
    (S,         'Shear S only',        'tab:orange'),
    (R @ S,     'R(Sx)\nShear → Rotate','tab:green'),
    (S @ R,     'S(Rx)\nRotate → Shear','tab:red'),
]

square = np.array([[0,0],[1,0],[1,1],[0,1],[0,0]]).T

for ax, (M, title, col) in zip(axes, configs):
    t = M @ square
    ax.fill(t[0], t[1], alpha=0.3, color=col)
    ax.plot(t[0], t[1], color=col, lw=2.5)
    # also draw original faintly
    ax.plot(square[0], square[1], 'k--', lw=1, alpha=0.4, label='original')
    ax.axhline(0, color='black', lw=0.7); ax.axvline(0, color='black', lw=0.7)
    ax.set_xlim(-2, 2.5); ax.set_ylim(-1.5, 2.5)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.2)
    ax.set_title(title, fontsize=9.5)

axes[2].set_title(f'R·S  (shear first, then rotate)\nAB', fontsize=9, color='tab:green')
axes[3].set_title(f'S·R  (rotate first, then shear)\nBA ≠ AB', fontsize=9, color='tab:red')

plt.tight_layout()
plt.savefig('diagram2_composition.png', bbox_inches='tight')
plt.show()

## Diagram 3 — Determinant as Signed Area
The determinant = signed area of the parallelogram formed by the column vectors. det=0 means the columns are collinear (space collapses).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Determinant = Signed Area of Parallelogram', fontsize=13, fontweight='bold')

cases = [
    (np.array([2, 0]), np.array([0, 2]),  'det = 4\n(2×2 square, positive)'),
    (np.array([1, 0]), np.array([1, 1]),  'det = 1\n(shear, area preserved)'),
    (np.array([2, 1]), np.array([4, 2]),  'det = 0\n(collinear → collapsed!)'),
]

colors = ['tab:blue', 'tab:green', 'tab:red']

for ax, (v1, v2, title), col in zip(axes, cases, colors):
    # parallelogram corners
    corners = np.array([[0,0], v1, v1+v2, v2, [0,0]])
    det_val = v1[0]*v2[1] - v1[1]*v2[0]
    
    ax.fill(corners[:,0], corners[:,1], alpha=0.25, color=col)
    ax.plot(corners[:,0], corners[:,1], color=col, lw=2)
    
    # draw column vectors
    ax.annotate('', xy=v1, xytext=(0,0),
                arrowprops=dict(arrowstyle='->', color='tab:red', lw=2.5))
    ax.annotate('', xy=v2, xytext=(0,0),
                arrowprops=dict(arrowstyle='->', color='tab:blue', lw=2.5))
    
    ax.text(v1[0]/2 - 0.3, v1[1]/2 - 0.2, f'col₁={list(v1)}', color='tab:red', fontsize=8)
    ax.text(v2[0]/2 + 0.1, v2[1]/2 + 0.1, f'col₂={list(v2)}', color='tab:blue', fontsize=8)
    ax.text(0.2, -0.4, f'det = {det_val:.0f}', fontsize=11, fontweight='bold', color=col)
    
    ax.axhline(0, color='black', lw=0.8); ax.axvline(0, color='black', lw=0.8)
    ax.set_xlim(-0.5, 6.5); ax.set_ylim(-0.8, 3.5)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.2)
    ax.set_title(title, fontsize=10)

plt.tight_layout()
plt.savefig('diagram3_determinant.png', bbox_inches='tight')
plt.show()
print('Red arrow = col₁, Blue arrow = col₂')
print('det=0 case: both columns point in same direction → parallelogram has zero area → space collapsed to a line')

## Diagram 4 — Change of Basis
Shows two coordinate systems (standard and Jenny's basis) describing the SAME vector with different numbers.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Change of Basis: Same Vector, Two Languages', fontsize=13, fontweight='bold')

# Standard basis
e1 = np.array([1, 0])
e2 = np.array([0, 1])

# Jenny's basis
b1 = np.array([2, 1])
b2 = np.array([-1, 1])
P  = np.column_stack([b1, b2])

# The vector in Jenny's coords = [3, -1]
jenny_coords = np.array([3, -1])
v_standard   = P @ jenny_coords   # = [7, 2] in standard

def draw_basis_grid(ax, b1, b2, color, alpha=0.18, n=5):
    for i in range(-n, n+1):
        start = -n*b2 + i*b1
        end   =  n*b2 + i*b1
        ax.plot([start[0], end[0]], [start[1], end[1]], color=color, alpha=alpha, lw=0.8)
        start = -n*b1 + i*b2
        end   =  n*b1 + i*b2
        ax.plot([start[0], end[0]], [start[1], end[1]], color=color, alpha=alpha, lw=0.8)

# LEFT: Standard basis view
ax = axes[0]
draw_basis_grid(ax, e1, e2, 'gray')

ax.annotate('', xy=e1*7, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='gray', lw=1.5, ls='--'))
ax.annotate('', xy=e2*2, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='gray', lw=1.5, ls='--'))

ax.annotate('', xy=v_standard, xytext=(0,0),
            arrowprops=dict(arrowstyle='->', color='tab:purple', lw=3))
ax.plot(*v_standard, 'o', color='tab:purple', ms=8)
ax.text(v_standard[0]+0.2, v_standard[1]+0.2, f'v = [7, 2]\n(standard coords)', color='tab:purple', fontsize=9)

ax.annotate('', xy=e1, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='black', lw=2))
ax.annotate('', xy=e2, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='black', lw=2))
ax.text(1.05, 0.05, 'e₁', fontsize=10, fontweight='bold')
ax.text(0.05, 1.05, 'e₂', fontsize=10, fontweight='bold')

ax.set_xlim(-2, 9); ax.set_ylim(-2, 5)
ax.set_aspect('equal'); ax.axhline(0,color='k',lw=0.7); ax.axvline(0,color='k',lw=0.7)
ax.set_title("Standard Basis\nv = 7·e₁ + 2·e₂ = [7, 2]", fontsize=10)
ax.grid(False)

# RIGHT: Jenny's basis view
ax = axes[1]
draw_basis_grid(ax, b1, b2, 'tab:orange')

ax.annotate('', xy=b1, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='tab:red', lw=2.5))
ax.annotate('', xy=b2, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='tab:blue', lw=2.5))
ax.text(b1[0]+0.1, b1[1]+0.1, 'b₁=[2,1]', color='tab:red', fontsize=9)
ax.text(b2[0]-0.8, b2[1]+0.1, 'b₂=[-1,1]', color='tab:blue', fontsize=9)

# draw 3·b1
ax.annotate('', xy=3*b1, xytext=(0,0),
            arrowprops=dict(arrowstyle='->', color='tab:red', lw=1.5, ls='dashed'))
# draw -1·b2 from 3b1
ax.annotate('', xy=3*b1 + (-1)*b2, xytext=3*b1,
            arrowprops=dict(arrowstyle='->', color='tab:blue', lw=1.5, ls='dashed'))

ax.annotate('', xy=v_standard, xytext=(0,0),
            arrowprops=dict(arrowstyle='->', color='tab:purple', lw=3))
ax.plot(*v_standard, 'o', color='tab:purple', ms=8)
ax.text(v_standard[0]+0.2, v_standard[1]+0.2, "v = [3, -1]\n(Jenny's coords)", color='tab:purple', fontsize=9)

ax.set_xlim(-3, 9); ax.set_ylim(-2, 5)
ax.set_aspect('equal'); ax.axhline(0,color='k',lw=0.7); ax.axvline(0,color='k',lw=0.7)
ax.set_title("Jenny's Basis\nv = 3·b₁ + (−1)·b₂ = [3,−1] in Jenny's language", fontsize=10)
ax.grid(False)

plt.tight_layout()
plt.savefig('diagram4_change_of_basis.png', bbox_inches='tight')
plt.show()
print('SAME purple vector — described as [7,2] in standard, [3,-1] in Jenny\'s basis')
print(f'Verify: P @ [3,-1] = {P @ jenny_coords}  ✓')

## Diagram 5 — Matrix Multiplication: 5 Perspectives Summary
Visual summary of how AB = C can be computed (all perspectives give the same result).

In [ ]:
A = np.array([[ 2, -1,  5],
              [ 3,  4,  4],
              [-4, -2,  0]])

B = np.array([[ 1,  0, -2],
              [ 1, -5,  1],
              [-3,  0,  3]])

C = A @ B

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Matrix Multiplication C = AB — Perspectives 1, 2, 4', fontsize=13, fontweight='bold')

def heatmap(ax, M, title, annot=True, cmap='RdBu_r', vmin=None, vmax=None):
    vm = np.abs(M).max() if vmin is None else abs(vmin)
    im = ax.imshow(M, cmap=cmap, vmin=-vm, vmax=vm, aspect='auto')
    if annot:
        for i in range(M.shape[0]):
            for j in range(M.shape[1]):
                ax.text(j, i, str(M[i,j]), ha='center', va='center',
                        fontsize=12, fontweight='bold',
                        color='white' if abs(M[i,j]) > vm*0.5 else 'black')
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title, fontsize=10)
    return im

# P1: highlight row 2 of A · col 1 of B = c21
ax = axes[0]
A_hl = A.astype(float); B_hl = B.astype(float); C_hl = C.astype(float)
heatmap(ax, C, 'Perspective 1: Rows × Cols\nc₂₁ = row2(A)·col1(B) = -5')
ax.add_patch(plt.Rectangle((-0.5, 1.5), 3, 1, fill=False, edgecolor='yellow', lw=3))
ax.add_patch(plt.Rectangle((-0.5, 1.5), 1, 1, fill=True, facecolor='yellow', alpha=0.4))
ax.set_title('Perspective 1: c_ij = row_i(A) · col_j(B)\n[highlighted: c₂₁ = -5]', fontsize=9)

# P2: show C[:,0] = A @ B[:,0]
ax = axes[1]
col_result = (A @ B[:,0]).reshape(-1,1)
C_p2 = C.copy()
heatmap(ax, C_p2, '')
ax.add_patch(plt.Rectangle((-0.5, -0.5), 1, 3, fill=True, facecolor='tab:green', alpha=0.3))
ax.add_patch(plt.Rectangle((-0.5, -0.5), 1, 3, fill=False, edgecolor='tab:green', lw=3))
ax.set_title('Perspective 2: C[:,j] = A · B[:,j]\n[col 1 of C = A × col 1 of B]', fontsize=9)

# P4: outer product sum visualization
ax = axes[2]
outer_sum = np.zeros((3,3))
for k in range(3):
    outer_sum += np.outer(A[:,k], B[k,:])
heatmap(ax, outer_sum.astype(int), '')
ax.set_title('Perspective 4: AB = Σₖ col_k(A)·row_k(B)\n[sum of 3 rank-1 outer products]', fontsize=9)

plt.tight_layout()
plt.savefig('diagram5_matmul_perspectives.png', bbox_inches='tight')
plt.show()
print('All three heatmaps are identical — same C, different ways of computing it.')

## Diagram 6 — Sandwich Formula: P⁻¹AP
Same transformation A, viewed in standard basis vs Jenny's basis. Both describe the same geometry.

In [ ]:
# Setup
b1 = np.array([2., 1.])
b2 = np.array([-1., 1.])
P  = np.column_stack([b1, b2])
Pinv = np.linalg.inv(P)

# Rotation 90 degrees in standard
A = np.array([[0., -1.], [1., 0.]])
M = Pinv @ A @ P   # same transformation in Jenny's basis

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
fig.suptitle("Sandwich Formula M = P⁻¹AP: Same Transform, Different Language",
             fontsize=12, fontweight='bold')

test_vectors = [np.array([1.,0.]), np.array([0.,1.]), np.array([1.,1.])]
colors_v = ['tab:red','tab:blue','tab:green']

for ax, (transform, title, basis_vecs, decode) in zip(axes, [
    (A, 'Standard Basis\nA = 90° rotation', [np.array([1.,0.]), np.array([0.,1.])], np.eye(2)),
    (M, "Jenny's Basis\nM = P⁻¹AP (same rotation)", [b1, b2], P),
]):
    # draw basis vectors
    bcolors = ['black', 'dimgray']
    blabels = ['b₁' if i==0 else 'b₂' for i in range(2)] if decode is not P else ['b₁','b₂']
    for i,(bv,bc) in enumerate(zip(basis_vecs, bcolors)):
        ax.annotate('', xy=bv, xytext=(0,0),
                    arrowprops=dict(arrowstyle='->', color=bc, lw=2))
        ax.text(bv[0]+0.05, bv[1]+0.05, blabels[i], fontsize=10, color=bc)

    for v, col in zip(test_vectors, colors_v):
        # original vector (in this basis)
        v_world = decode @ v
        # transformed vector
        if decode is np.eye(2):
            tv_world = A @ v_world
        else:
            # transform in Jenny's language, decode to world
            tv_world = A @ v_world  # same geometry!
        
        ax.annotate('', xy=v_world, xytext=(0,0),
                    arrowprops=dict(arrowstyle='->', color=col, lw=2, alpha=0.5))
        ax.annotate('', xy=tv_world, xytext=(0,0),
                    arrowprops=dict(arrowstyle='->', color=col, lw=2.5,
                                   connectionstyle='arc3,rad=0.3'))
        ax.plot([v_world[0], tv_world[0]], [v_world[1], tv_world[1]],
                '--', color=col, alpha=0.4, lw=1)

    ax.set_xlim(-3, 3.5); ax.set_ylim(-2.5, 3.5)
    ax.set_aspect('equal')
    ax.axhline(0, color='black', lw=0.7); ax.axvline(0, color='black', lw=0.7)
    ax.grid(True, alpha=0.2)
    ax.set_title(title, fontsize=10)

# Print matrix values
print(f'A (standard) =\n{A}')
print(f'M = P⁻¹AP (Jenny) =\n{np.round(M, 3)}')
print(f'\ndet(A) = {np.linalg.det(A):.2f},  det(M) = {np.linalg.det(M):.2f}  ← same!')
print(f'eigenvalues of A: {np.round(np.linalg.eigvals(A),3)}')
print(f'eigenvalues of M: {np.round(np.linalg.eigvals(M),3)}  ← same!')

plt.tight_layout()
plt.savefig('diagram6_sandwich.png', bbox_inches='tight')
plt.show()

## Summary

| Diagram | Key Insight |
|---|---|
| 1 — Transformations | Columns of A = where basis vectors land |
| 2 — Composition | AB ≠ BA — rightmost acts first |
| 3 — Determinant | Signed area of parallelogram; det=0 collapses space |
| 4 — Change of Basis | Same vector, two coordinate languages |
| 5 — Matmul Perspectives | 5 ways to compute AB, all equivalent |
| 6 — Sandwich P⁻¹AP | Same geometry, different language; eigenvalues preserved |